# Load Data

In [1]:
# %%writefile part_d.py
import numpy as np
import pandas as pd
import pickle
import os
import sys
from pathlib import Path

from sklearn.linear_model import RidgeCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from scipy.signal import find_peaks

import neurokit2 as nk

In [2]:
# %%writefile -a part_d.py
FORMAT_VERSION = 1

# Feature engineering
Single source of truth for turning one raw `.npz` window into a feature row. Used identically by both `train` and `feature_engineering` modes.

In [3]:
# %%writefile -a part_d.py

def _stats(x, prefix, names):
    """
    Basic summary stats for a 1D array, ignoring NaNs. Returns values + appends names.
    If a window has no valid samples at all for this modality, emits np.nan rather
    than a hardcoded fallback (e.g. 0.0) — a fabricated constant would silently
    distort downstream model fitting/interpretation, whereas np.nan is handled
    explicitly and only ever imputed using train-set statistics (see cmd_train).
    """
    x = x[~np.isnan(x)]
    if x.size == 0:
        vals = [np.nan, np.nan, np.nan, np.nan, np.nan]
    else:
        vals = [
            float(np.mean(x)),
            float(np.std(x)),
            float(np.min(x)),
            float(np.max(x)),
            float(np.max(x) - np.min(x)),
        ]
    names.extend([f"{prefix}_mean", f"{prefix}_std", f"{prefix}_min", f"{prefix}_max", f"{prefix}_range"])
    return vals

def peak_intervals(x, fs, min_rate_per_min, max_rate_per_min, prominence=None):
    """
    Returns peak indices and inter-peak intervals (seconds).
    `prominence` rejects peaks that don't rise clearly above the local
    noise floor -- without it, `distance` alone lets small noise wiggles
    (e.g. T-waves, sensor noise) count as full peaks and silently double
    (or more) the detected event rate.
    """
    min_distance = int(fs * 60 / max_rate_per_min)
    peaks, _ = find_peaks(x, distance=min_distance, prominence=prominence)
    intervals = np.diff(peaks) / fs
    return peaks, intervals


def _slope(x, prefix, names):
    """Least-squares slope of a 1D array against sample index, ignoring NaNs."""
    idx = np.arange(x.shape[0])
    mask = ~np.isnan(x)
    if mask.sum() < 2:
        val = np.nan
    else:
        val = float(np.polyfit(idx[mask], x[mask], 1)[0])
    names.append(f"{prefix}_slope")
    return [val]


def _breathing_rate_features(x, fs, names, min_rate_per_min=6, max_rate_per_min=40):
    """
    Respiration-rate features from the raw breathing waveform: instantaneous
    breathing rate (from peak-to-peak timing) and breath amplitude, both
    mean/std across the window.

    NaN-safe: interpolates over isolated missing samples (the signal is
    smooth/periodic, so linear interpolation is reasonable); if too much of
    the window is missing, or too few breaths are detected to form an
    interval, emits np.nan so the downstream imputer (fit on train only)
    handles it consistently with every other feature.
    """
    prefix = "breathing"
    feature_names = [f"{prefix}_resp_rate", f"{prefix}_resp_rate_std",
                      f"{prefix}_amp_mean", f"{prefix}_amp_std"]

    x = np.asarray(x, dtype=np.float64)
    valid = ~np.isnan(x)

    if valid.sum() < 0.5 * valid.size:
        names.extend(feature_names)
        return [np.nan, np.nan, np.nan, np.nan]

    idx = np.arange(x.size)
    x_filled = np.interp(idx, idx[valid], x[valid])
    prominence = 0.5 * np.std(x_filled)

    peaks, intervals = peak_intervals(x_filled, fs=fs,
                                       min_rate_per_min=min_rate_per_min,
                                       max_rate_per_min=max_rate_per_min,
                                       prominence=prominence)

    if intervals.size < 2:
        names.extend(feature_names)
        return [np.nan, np.nan, np.nan, np.nan]

    inst_rate = 60.0 / intervals  # breaths per minute, per interval
    resp_rate = float(np.mean(inst_rate))
    resp_rate_std = float(np.std(inst_rate))

    # Amplitude: peak height above the nearest preceding trough, per breath.
    min_distance = max(1, int(fs * 60 / max_rate_per_min))
    troughs, _ = find_peaks(-x_filled, distance=min_distance, prominence=prominence)
    amps = []
    for p in peaks:
        prior_troughs = troughs[troughs < p]
        if prior_troughs.size:
            amps.append(x_filled[p] - x_filled[prior_troughs[-1]])

    amp_mean = float(np.mean(amps)) if amps else np.nan
    amp_std = float(np.std(amps)) if amps else np.nan

    names.extend(feature_names)
    return [resp_rate, resp_rate_std, amp_mean, amp_std]


def _interpolate_nans(x):
    """Linear interpolation over isolated missing samples. Returns None if
    too much of the signal is missing to interpolate meaningfully."""
    x = np.asarray(x, dtype=np.float64)
    valid = ~np.isnan(x)
    if valid.sum() < 0.5 * valid.size:
        return None
    idx = np.arange(x.size)
    return np.interp(idx, idx[valid], x[valid])


def _moving_average(x, window):
    """
    Edge-padded moving average. np.convolve(x, kernel, mode="same") looks
    tempting here but implicitly zero-pads outside the signal -- fine for a
    zero-centered signal, but catastrophic for something like raw ECG counts
    with a large nonzero baseline (~2,000,000): near the edges, the kernel
    averages in implicit zeros, so the baseline estimate collapses toward
    zero right where we need it most, leaving a huge residual offset after
    subtraction. Edge-padding the input before convolving keeps the
    baseline estimate consistent across the whole window, edges included.
    """
    if window <= 1:
        return x.copy()
    pad = window // 2
    padded = np.pad(x, pad, mode="edge")
    kernel = np.ones(window) / window
    baseline = np.convolve(padded, kernel, mode="valid")
    return baseline[:x.size]


def _detrend(x, fs, baseline_window_s=30):
    """
    Removes a slow moving-average baseline from a raw signal, isolating
    faster within-window signal shape from the sensor's arbitrary
    calibration offset/drift. The raw zephyr_ecg/zephyr_breathing channels
    here are unscaled device counts with large, subject-specific DC offsets
    (e.g. breathing_mean ~ 4,100,000) that reflect electrode contact/strap
    fit -- not physiology -- and were found to correlate with glucose only
    because they act as a subject fingerprint (see the d1 vs d2 correlation
    comparison). Detrending removes that offset before any level-based stat
    is computed on these two channels.
    """
    x_filled = _interpolate_nans(x)
    if x_filled is None:
        return None
    window = max(1, int(baseline_window_s * fs))
    if window >= x_filled.size:
        return x_filled - np.mean(x_filled)
    baseline = _moving_average(x_filled, window)
    return x_filled - baseline


def _stats_detrended(x, fs, prefix, names, baseline_window_s=30):
    """Like _stats, but computed on the detrended signal (see _detrend) so
    mean/min/max/range reflect real within-window signal shape rather than
    the sensor's arbitrary calibration offset."""
    x_detrended = _detrend(x, fs, baseline_window_s=baseline_window_s)
    if x_detrended is None:
        vals = [np.nan, np.nan, np.nan, np.nan, np.nan]
    else:
        vals = [
            float(np.mean(x_detrended)),
            float(np.std(x_detrended)),
            float(np.min(x_detrended)),
            float(np.max(x_detrended)),
            float(np.max(x_detrended) - np.min(x_detrended)),
        ]
    names.extend([f"{prefix}_mean", f"{prefix}_std", f"{prefix}_min", f"{prefix}_max", f"{prefix}_range"])
    return vals


def _block_trend_features(x, prefix, names, n_blocks=10):
    """
    Splits the window into n_blocks equal chunks and reports how much the
    signal moved from the first half of the window to the second half
    (block_trend), plus how much block-to-block variability there is
    (block_var). Complements the whole-window linear _slope: a window that
    dips then recovers has ~zero net slope but real block_var, which a
    single slope value can't distinguish from a flat window -- and acute
    autonomic responses are often about *when* something changes, not just
    the net drift over 5 minutes.
    """
    feature_names = [f"{prefix}_block_trend", f"{prefix}_block_var"]
    x = np.asarray(x, dtype=np.float64)
    valid = ~np.isnan(x)

    if valid.sum() < 0.5 * valid.size:
        names.extend(feature_names)
        return [np.nan, np.nan]

    idx = np.arange(x.size)
    x_filled = np.interp(idx, idx[valid], x[valid])

    blocks = np.array_split(x_filled, n_blocks)
    block_means = np.array([np.mean(b) for b in blocks])

    half = n_blocks // 2
    block_trend = float(np.mean(block_means[half:]) - np.mean(block_means[:half]))
    block_var = float(np.std(block_means))

    names.extend(feature_names)
    return [block_trend, block_var]


def _hrv_features(x, fs, names):
    """
    Heart-rate variability features from ECG, via NeuroKit2's validated
    R-peak detector/HRV pipeline (used unconditionally -- no hand-rolled
    fallback): R-peak-derived heart rate, SDNN (overall RR-interval
    variability) and RMSSD (beat-to-beat variability). RMSSD in particular
    is sensitive to sympathetic/parasympathetic shifts -- the same
    autonomic response that acute glucose swings (especially hypoglycemia)
    trigger -- unlike plain amplitude stats on the raw ECG trace.

    Wrapped in a try/except: NeuroKit2 can raise on very short/degenerate
    windows it can't find a usable QRS complex in, in which case this
    emits np.nan like every other feature here, rather than crashing the
    whole feature-extraction pass.
    """
    prefix = "ecg"
    feature_names = [f"{prefix}_hr_mean", f"{prefix}_sdnn", f"{prefix}_rmssd"]

    x_filled = _interpolate_nans(x)
    if x_filled is None:
        names.extend(feature_names)
        return [np.nan, np.nan, np.nan]

    try:
        _, info = nk.ecg_peaks(x_filled, sampling_rate=fs)
        hrv = nk.hrv_time(info, sampling_rate=fs, show=False)
        mean_nn = float(hrv["HRV_MeanNN"].iloc[0])
        sdnn_ms = float(hrv["HRV_SDNN"].iloc[0])
        rmssd_ms = float(hrv["HRV_RMSSD"].iloc[0])
        if not np.isfinite(mean_nn) or mean_nn <= 0:
            raise ValueError("no usable NN intervals")
        hr_mean = 60000.0 / mean_nn
        sdnn = sdnn_ms / 1000.0   # ms -> s, matching the rest of this file's units
        rmssd = rmssd_ms / 1000.0
    except Exception:
        names.extend(feature_names)
        return [np.nan, np.nan, np.nan]

    names.extend(feature_names)
    return [float(hr_mean), float(sdnn), float(rmssd)]


def _eda_scr_features(x, fs, names, tonic_window_s=20, min_amp_rel=0.5):
    """
    Skin-conductance-response (SCR) event features from EDA: how many
    sudden phasic spikes occurred in the window, and how large they were.
    Sweating (adrenergic sweat-gland activation) is a textbook acute
    hypoglycemia symptom, so SCR *events* -- not the mean EDA level -- are
    the physiologically motivated signal here.

    Tonic (slow baseline) component is estimated with a moving average and
    subtracted to isolate the phasic (fast) component, in which peaks are
    detected as SCR events. NaN-safe like the other peak-based features.
    """
    prefix = "eda"
    feature_names = [f"{prefix}_scr_count", f"{prefix}_scr_amp_mean", f"{prefix}_scr_amp_sum"]

    x_filled = _interpolate_nans(x)
    if x_filled is None:
        names.extend(feature_names)
        return [np.nan, np.nan, np.nan]

    window = max(1, int(tonic_window_s * fs))
    tonic = _moving_average(x_filled, window)
    phasic = x_filled - tonic

    # Relative threshold: EDA's absolute scale varies hugely by person/device
    # calibration, so a fixed absolute height cuts off very differently across
    # subjects. Scale to this window's own phasic variability instead, with a
    # tiny floor so a near-flat phasic signal doesn't trigger on pure noise.
    min_distance = max(1, int(fs * 1.0))  # SCRs don't repeat faster than ~1/s
    min_amp = max(1e-3, min_amp_rel * np.std(phasic))
    scr_peaks, _ = find_peaks(phasic, distance=min_distance, height=min_amp)

    scr_count = float(scr_peaks.size)
    if scr_peaks.size == 0:
        amp_mean, amp_sum = 0.0, 0.0
    else:
        amps = phasic[scr_peaks]
        amp_mean = float(np.mean(amps))
        amp_sum = float(np.sum(amps))

    names.extend(feature_names)
    return [scr_count, amp_mean, amp_sum]


def _bvp_pulse_features(x, fs, names, min_rate_per_min=40, max_rate_per_min=200):
    """
    Pulse-morphology features from BVP: pulse-rate variability (PRV, the
    PPG analogue of HRV), mean systolic rise time, and pulse amplitude.
    Vascular tone/blood viscosity shifts (glucose- and adrenaline-linked)
    subtly change how sharply and how strongly each pulse rises, which
    plain waveform stats don't capture.

    NaN-safe like the other peak-based features.
    """
    prefix = "bvp"
    feature_names = [f"{prefix}_prv_sdnn", f"{prefix}_prv_rmssd",
                      f"{prefix}_rise_time_mean", f"{prefix}_pulse_amp_mean"]

    x_filled = _interpolate_nans(x)
    if x_filled is None:
        names.extend(feature_names)
        return [np.nan, np.nan, np.nan, np.nan]

    min_distance = max(1, int(fs * 60 / max_rate_per_min))
    prominence = 0.5 * np.std(x_filled)
    peaks, intervals = peak_intervals(x_filled, fs=fs,
                                       min_rate_per_min=min_rate_per_min,
                                       max_rate_per_min=max_rate_per_min,
                                       prominence=prominence)

    if intervals.size < 2:
        names.extend(feature_names)
        return [np.nan, np.nan, np.nan, np.nan]

    prv_sdnn = float(np.std(intervals))
    prv_rmssd = float(np.sqrt(np.mean(np.diff(intervals) ** 2)))

    troughs, _ = find_peaks(-x_filled, distance=min_distance, prominence=prominence)
    rise_times, amps = [], []
    for p in peaks:
        prior_troughs = troughs[troughs < p]
        if prior_troughs.size:
            t = prior_troughs[-1]
            rise_times.append((p - t) / fs)
            amps.append(x_filled[p] - x_filled[t])

    rise_time_mean = float(np.mean(rise_times)) if rise_times else np.nan
    pulse_amp_mean = float(np.mean(amps)) if amps else np.nan

    names.extend(feature_names)
    return [prv_sdnn, prv_rmssd, rise_time_mean, pulse_amp_mean]


def _ptt_features(ecg, bvp, fs_ecg, fs_bvp, names,
                   min_rate_per_min=40, max_rate_per_min=200):
    """
    Pulse transit time (PTT): the delay between each ECG R-peak (electrical
    onset of a heartbeat) and the BVP peak it produces at the periphery.
    PTT is a validated proxy for arterial stiffness/vascular tone -- a more
    direct mechanistic link to glucose- and adrenaline-driven vascular
    effects than either signal's HRV/PRV alone, since it directly measures
    how fast the pulse wave travels rather than just how regularly it
    repeats.

    NaN-safe like the other peak-based features. Implausible gaps (>1s,
    suggesting a missed/mismatched beat) are discarded per-pair rather than
    corrupting the mean.
    """
    prefix = "ptt"
    feature_names = [f"{prefix}_mean", f"{prefix}_std"]

    ecg_filled = _interpolate_nans(ecg)
    bvp_filled = _interpolate_nans(bvp)
    if ecg_filled is None or bvp_filled is None:
        names.extend(feature_names)
        return [np.nan, np.nan]

    ecg_peaks, _ = peak_intervals(ecg_filled, fs=fs_ecg,
                                   min_rate_per_min=min_rate_per_min,
                                   max_rate_per_min=max_rate_per_min,
                                   prominence=0.5 * np.std(ecg_filled))
    bvp_peaks, _ = peak_intervals(bvp_filled, fs=fs_bvp,
                                   min_rate_per_min=min_rate_per_min,
                                   max_rate_per_min=max_rate_per_min,
                                   prominence=0.5 * np.std(bvp_filled))

    if ecg_peaks.size < 2 or bvp_peaks.size < 2:
        names.extend(feature_names)
        return [np.nan, np.nan]

    ecg_times = ecg_peaks / fs_ecg
    bvp_times = bvp_peaks / fs_bvp

    ptts = []
    for t_r in ecg_times:
        later = bvp_times[bvp_times > t_r]
        if later.size:
            gap = later[0] - t_r
            if gap < 1.0:  # reject implausible pairings (missed beat etc.)
                ptts.append(gap)

    if len(ptts) < 2:
        names.extend(feature_names)
        return [np.nan, np.nan]

    names.extend(feature_names)
    return [float(np.mean(ptts)), float(np.std(ptts))]


def make_features_for_window(record, feature_names_out):
    """
    record: dict-like with per-window 1D/2D arrays already sliced for ONE example,
            e.g. record["e4_bvp"] has shape (19200,), record["e4_acc"] has shape (9600, 3).
    feature_names_out: list to append feature names to (only populated on first call;
                        caller is responsible for only using this on the first row and
                        asserting consistency afterwards).
    Returns: 1D numpy array of feature values for this example.
    """
    names = []
    vals = []

    # --- BVP (E4 PPG) ---
    bvp = record["e4_bvp"]
    vals += _stats(bvp, "bvp", names)
    vals += _slope(bvp, "bvp", names)
    vals += _bvp_pulse_features(bvp, fs=64, names=names)

    # --- E4 HR (device-derived heart rate trace) ---
    hr = record["e4_hr"]
    vals += _stats(hr, "e4_hr", names)
    vals += _block_trend_features(hr, "e4_hr", names)

    # --- EDA ---
    eda = record["e4_eda"]
    vals += _stats(eda, "eda", names)
    vals += _slope(eda, "eda", names)
    vals += _eda_scr_features(eda, fs=4, names=names)
    vals += _block_trend_features(eda, "eda", names)

    # --- Temperature ---
    temp = record["e4_temp"]
    vals += _stats(temp, "temp", names)
    vals += _slope(temp, "temp", names)
    vals += _block_trend_features(temp, "temp", names)

    # --- E4 accelerometer: squared magnitude a_sq(t) = ax^2+ay^2+az^2 ---
    acc = record["e4_acc"]  # (T, 3)
    acc_sq = np.nansum(acc.astype(np.float64) ** 2, axis=1)
    vals += _stats(acc_sq, "e4_acc_sq", names)

    # --- Zephyr ECG ---
    ecg = record["zephyr_ecg"]
    vals += _stats_detrended(ecg, fs=250, prefix="ecg", names=names)
    vals += _hrv_features(ecg, fs=250, names=names)
    vals += _ptt_features(ecg, bvp, fs_ecg=250, fs_bvp=64, names=names)

    # --- Zephyr accelerometer magnitude ---
    zacc = record["zephyr_acc"]
    zacc_sq = np.nansum(zacc.astype(np.float64) ** 2, axis=1)
    vals += _stats(zacc_sq, "zephyr_acc_sq", names)

    # --- Zephyr breathing ---
    breathing = record["zephyr_breathing"]
    vals += _stats_detrended(breathing, fs=25, prefix="breathing", names=names)
    vals += _breathing_rate_features(breathing, fs=25, names=names)

    if not feature_names_out:
        feature_names_out.extend(names)
    else:
        assert feature_names_out == names, "Feature name/order mismatch between windows"

    return np.array(vals, dtype=np.float64)

# Streaming file processing
Processes one `.npz` file at a time to keep memory bounded (indexing into an npz array still loads the full array first, so we must avoid holding many files in memory at once).

In [5]:
# %%writefile -a part_d.py

def iter_participant_files(data_dir):
    return sorted(Path(data_dir).glob("*.npz"))


def build_feature_matrix(data_dir, has_target):
    """
    Processes one .npz file at a time to keep memory bounded.
    Returns (Z, y, feature_names) where y is None if has_target is False.
    """
    feature_names = []
    feature_rows = []
    targets = [] if has_target else None

    for path in iter_participant_files(data_dir):
        with np.load(path, allow_pickle=False) as data:
            n = data["e4_bvp"].shape[0]

            fields = {
                "e4_bvp": data["e4_bvp"],
                "e4_hr": data["e4_hr"],
                "e4_eda": data["e4_eda"],
                "e4_temp": data["e4_temp"],
                "e4_acc": data["e4_acc"],
                "zephyr_ecg": data["zephyr_ecg"],
                "zephyr_acc": data["zephyr_acc"],
                "zephyr_breathing": data["zephyr_breathing"],
            }

            if has_target:
                glucose = data["glucose"]

            for i in range(n):
                record = {k: v[i] for k, v in fields.items()}
                row = make_features_for_window(record, feature_names)
                feature_rows.append(row)

            if has_target:
                targets.append(glucose)

    Z = np.vstack(feature_rows)
    y = np.concatenate(targets) if has_target else None
    return Z, y, feature_names

# Train / feature_engineering entry points

In [6]:
# %%writefile -a part_d.py

def cmd_train(protocol, train_dir, model_path, show_correlations=False, top_n=15):
    Z, y, feature_names = build_feature_matrix(train_dir, has_target=True)

    if show_correlations:
        df = pd.DataFrame(Z, columns=feature_names)
        df["glucose"] = y
        corr = df.corr(numeric_only=True)["glucose"].drop("glucose")
        corr = corr.reindex(corr.abs().sort_values(ascending=False).index)
        print(f"\n=== feature-glucose correlation (top {top_n}) ===")
        for name, r in corr.head(top_n).items():
            print(f"{name:25s} {r:8.4f}")
        print()

    imputer = SimpleImputer(strategy="median")
    Z_imputed = imputer.fit_transform(Z)

    scaler = StandardScaler()
    Z_scaled = scaler.fit_transform(Z_imputed)

    model = RidgeCV(alphas=np.logspace(-3, 6, 19))
    model.fit(Z_scaled, y)

    # Convert standardized-space coefficients back to raw feature scale so that
    # saved (intercept, coef) act directly on the *unscaled* engineered features.
    # y_hat = intercept_s + coef_s . ((z - mean) / scale)
    #       = (intercept_s - sum(coef_s * mean / scale)) + sum((coef_s/scale) * z)
    coef_scaled = model.coef_
    coef_raw = coef_scaled / scaler.scale_
    intercept_raw = model.intercept_ - np.sum(coef_scaled * scaler.mean_ / scaler.scale_)

    state = {
        "format_version": FORMAT_VERSION,
        "protocol": protocol,
        "intercept": float(intercept_raw),
        "coef": coef_raw.astype(np.float64),
        "feature_names": feature_names,
        "preprocessing_state": {
            "imputer": imputer,
        },
    }

    with open(model_path, "wb") as f:
        pickle.dump(state, f)

    print(f"[train] protocol={protocol} n={Z.shape[0]} m={Z.shape[1]} "
          f"best_alpha={model.alpha_:.4g}")


def cmd_feature_engineering(protocol, test_dir, model_path, output_path):
    with open(model_path, "rb") as f:
        state = pickle.load(f)

    assert state["protocol"] == protocol, "Model/protocol mismatch"

    Z, _, feature_names = build_feature_matrix(test_dir, has_target=False)
    assert feature_names == state["feature_names"], "Feature name/order mismatch vs. training"

    imputer = state["preprocessing_state"]["imputer"]
    Z_imputed = imputer.transform(Z)

    assert np.isfinite(Z_imputed).all(), "Non-finite values remain in final feature matrix"

    np.save(output_path, Z_imputed)
    print(f"[feature_engineering] protocol={protocol} n={Z_imputed.shape[0]} m={Z_imputed.shape[1]}")

# CLI

In [7]:
# %%writefile -a part_d.py

def main():
    if len(sys.argv) < 2:
        print("usage:\n"
              "  part_d.py train <protocol> <train_dir> <model_path>\n"
              "  part_d.py feature_engineering <protocol> <test_dir> <model_path> <output_path>")
        sys.exit(1)

    mode = sys.argv[1]

    if mode == "train":
        _, _, protocol, train_dir, model_path = sys.argv
        cmd_train(protocol, train_dir, model_path)
    elif mode == "feature_engineering":
        _, _, protocol, test_dir, model_path, output_path = sys.argv
        cmd_feature_engineering(protocol, test_dir, model_path, output_path)
    else:
        raise ValueError(f"Unknown mode: {mode}")


# if __name__ == "__main__":
#     main()

# Dev / Testing on Kaggle

Everything below this point is **dev-only** — none of it is part of the submitted `part_d.py` (nothing here has a `%%writefile` marker). It's for running and validating d1/d2/d3 locally before submission.

## Config

Single source of truth for every protocol's paths. `PROTOCOLS[name]` gives `train_dir`/`test_dir`/`model_path`/`features_path` for `"d1"`, `"d2"`, `"d3"` — every cell below should read from this dict rather than ad hoc globals, so there's no way to silently train on the wrong protocol's data.

d3 needs disjoint-participant folders built from `train_set`/`test_set` (which as shipped are laid out for **d2** — same participant, earlier vs. later segment). The `subject_train`/`subject_test` split below mirrors the mapping in `PartdData/Data_organization.md`, and is applied here via symlinks into `/kaggle/working/` (input is read-only, and these files are large, so we link rather than copy).

In [8]:
base_dir = "./PartdData"

train_set = f"{base_dir}/train_set"
test_set = f"{base_dir}/test_set"
train_d3 = f"{base_dir}/train_d3"
test_d3 = f"{base_dir}/test_d3"

# Per Data_organization.md: these 8 participants are entirely train in d3,
# these 2 are entirely test -- both their _a and _b files move accordingly.
subject_train = ["c1s01", "c1s02", "c1s03", "c1s05", "c2s01", "c2s02", "c2s04", "c2s05"]
subject_test = ["c1s04", "c2s03"]

def safe_symlink(src, dst):
    if os.path.lexists(dst):  # lexists also catches broken symlinks
        os.remove(dst)
    os.symlink(src, dst)

os.makedirs(train_d3, exist_ok=True)
os.makedirs(test_d3, exist_ok=True)

for s in subject_train:
    safe_symlink(f"{train_set}/{s}_a.npz", f"{train_d3}/{s}_a.npz")
    safe_symlink(f"{test_set}/{s}_b.npz",  f"{train_d3}/{s}_b.npz")

for s in subject_test:
    safe_symlink(f"{train_set}/{s}_a.npz", f"{test_d3}/{s}_a.npz")
    safe_symlink(f"{test_set}/{s}_b.npz",  f"{test_d3}/{s}_b.npz")

PROTOCOLS = {
    "d1": dict(
        train_dir=f"{base_dir}/random_train",
        test_dir=f"{base_dir}/random_test",
        model_path="model_d1.pkl",
        features_path="features_d1.npy",
    ),
    "d2": dict(
        train_dir=train_set,
        test_dir=test_set,
        model_path="model_d2.pkl",
        features_path="features_d2.npy",
    ),
    "d3": dict(
        train_dir=train_d3,
        test_dir=test_d3,
        model_path="model_d3.pkl",
        features_path="features_d3.npy",
    ),
}

## Real CLI equivalent

What the grader actually runs, for reference -- uncomment to sanity-check `part_d.py` itself (as written to disk by the `%%writefile` cells above) rather than the in-notebook functions.

In [ ]:
# for name, cfg in PROTOCOLS.items():
#     get_ipython().system(
#         f'python3 part_d.py train {name} {cfg["train_dir"]} {cfg["model_path"]}'
#     )
#     get_ipython().system(
#         f'python3 part_d.py feature_engineering {name} {cfg["test_dir"]} '
#         f'{cfg["model_path"]} {cfg["features_path"]}'
#     )


## Local evaluation (dev only)

Uses the dev test folders' `glucose` field (the real hidden test set won't have it) to compute NMAE/NMSE against your model and a median-training-target baseline, for the report's "Median baseline comparison" section.

In [9]:
def nmae(y_true, y_pred):
    y_mean = np.mean(y_true)
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true - y_mean))


def nmse(y_true, y_pred):
    y_mean = np.mean(y_true)
    return np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_mean) ** 2)


def local_eval(protocol, train_dir, test_dir, model_path, show_correlations=True, top_n=15):
    # Train (writes model_path), exactly as the real CLI would.
    cmd_train(protocol, train_dir, model_path, show_correlations=show_correlations, top_n=top_n)
    print("Training Done\n\n")

    with open(model_path, "rb") as f:
        state = pickle.load(f)

    # Median baseline needs y_train.
    _, y_train, _ = build_feature_matrix(train_dir, has_target=True)
    median_pred = np.median(y_train)

    # Dev test folders include glucose (unlike the real hidden test set) so we
    # can self-evaluate. build_feature_matrix with has_target=True reads it.
    Z_test, y_test, feature_names = build_feature_matrix(test_dir, has_target=True)
    assert feature_names == state["feature_names"], "Feature name/order mismatch vs. training"

    imputer = state["preprocessing_state"]["imputer"]
    Z_test_imputed = imputer.transform(Z_test)
    assert np.isfinite(Z_test_imputed).all(), "Non-finite values in test feature matrix"

    y_pred = state["intercept"] + Z_test_imputed @ state["coef"]
    y_pred_median = np.full_like(y_test, median_pred)

    print(f"\n=== Protocol {protocol} ===")
    print(f"n_train={len(y_train)}  n_test={len(y_test)}  n_features={len(feature_names)}")
    print(f"median(y_train) = {median_pred:.2f} mg/dL")
    print()
    print(f"{'':12s} {'NMAE':>10s} {'NMSE':>10s}")
    print(f"{'model':12s} {nmae(y_test, y_pred):10.4f} {nmse(y_test, y_pred):10.4f}")
    print(f"{'median':12s} {nmae(y_test, y_pred_median):10.4f} {nmse(y_test, y_pred_median):10.4f}")

    return state, y_test, y_pred


In [10]:
# Run one protocol at a time, e.g.:
cfg = PROTOCOLS["d1"]
local_eval("d1", cfg["train_dir"], cfg["test_dir"], cfg["model_path"])

# Or all three in one go:
# for name, cfg in PROTOCOLS.items():
#     local_eval(name, cfg["train_dir"], cfg["test_dir"], cfg["model_path"])


/opt/miniconda3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/opt/miniconda3/lib/python3.13/site-packages/neurokit2/hrv/hrv_time.py:165: RuntimeWarning: Mean of empty slice
  out["MeanNN"] = np.nanmean(rri)
/opt/miniconda3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/miniconda3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/opt/miniconda3/lib/python3.13/site-packages/neurokit2/hrv/hrv_time.py:165: RuntimeWarning: Mean of empty slice
  out["MeanNN"] = np.nanmean(rri)
/opt/miniconda3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axi

KeyboardInterrupt: 

## Inspect a trained model (dev only)

Pulls the per-feature training-set median straight out of the pickled `SimpleImputer` -- a cheap sanity check on feature plausibility (e.g. is `ecg_hr_mean` a real heart rate?) without needing to reload the raw data.

In [ ]:
def inspect_model(model_path):
    with open(model_path, "rb") as f:
        state = pickle.load(f)

    imputer = state["preprocessing_state"]["imputer"]
    for name, median in zip(state["feature_names"], imputer.statistics_):
        print(f"{name:25s} median={median:.4f}")

    return state


inspect_model(PROTOCOLS["d1"]["model_path"])
inspect_model(PROTOCOLS["d2"]["model_path"])
inspect_model(PROTOCOLS["d3"]["model_path"])


## Feature-target correlation diagnostic (dev only)

Quick single-feature Pearson correlation against `glucose`, computed pairwise (NaN-safe via pandas) so this can run before any imputation. Use this per protocol to see which features carry real signal vs. which are likely subject-identity/calibration leakage (compare d1/d2 vs. d3).

In [ ]:
def feature_correlations(train_dir, top_n=15):
    Z, y, feature_names = build_feature_matrix(train_dir, has_target=True)

    df = pd.DataFrame(Z, columns=feature_names)
    df["glucose"] = y

    corr = df.corr(numeric_only=True)["glucose"].drop("glucose")
    corr = corr.reindex(corr.abs().sort_values(ascending=False).index)

    print(f"n={len(y)}  n_features={len(feature_names)}")
    print()
    print(f"{'feature':25s} {'corr':>8s}")
    for name, r in corr.head(top_n).items():
        print(f"{name:25s} {r:8.4f}")

    return corr


# corr_d1 = feature_correlations(PROTOCOLS["d1"]["train_dir"])
# corr_d2 = feature_correlations(PROTOCOLS["d2"]["train_dir"])
# corr_d3 = feature_correlations(PROTOCOLS["d3"]["train_dir"])
